In [11]:
import ipywidgets as widgets
widgets.IntText(value=42, description='Test:')

IntText(value=42, description='Test:')

RuntimeError: <bound method FigureCanvasAgg.print_png of <matplotlib.backends.backend_agg.FigureCanvasAgg object at 0xa9e01088>> did not call Figure.draw, so no renderer is available

In [9]:
from pynq import Overlay
import numpy as np

ol = Overlay('/home/xilinx/angus.bit')
print(ol.ip_dict.keys())  # should show your IP blocks

import ipywidgets as widgets
from IPython.display import display
import threading
import time

ol   = Overlay('/home/xilinx/angus.bit')
regs = ol.top_0

# Initial config
regs.write(0x04, 192)    # GAP_THRESH
regs.write(0x20, 1)      # DECIMATION
regs.write(0x00, 0x03)   # CONTROL enable

dict_keys(['top_0', 'axi_dma_0', 'processing_system7_0'])


In [10]:
STATES = {0:'UNSYNC', 1:'FIRST_GAP', 2:'SYNC_CRANK', 3:'SYNC_FULL'}

# --- Parameter widgets ---
w_kp    = widgets.BoundedIntText(value=256,  min=0, max=65535, description='KP:')
w_ki    = widgets.BoundedIntText(value=16,   min=0, max=65535, description='KI:')
w_maxc  = widgets.BoundedIntText(value=1024, min=0, max=65535, description='Max corr:')
w_phase = widgets.BoundedIntText(value=1717, min=0, max=7199,  description='Phase ang:')
w_tol   = widgets.BoundedIntText(value=300,  min=0, max=7199,  description='Phase tol:')
w_write = widgets.Button(description='Write all', button_style='primary')

# --- Status widgets ---
w_state  = widgets.Label(value='sync: —')
w_rpm    = widgets.Label(value='RPM: —')
w_ph_err = widgets.Label(value='phase err: —')
w_corr   = widgets.Label(value='correction: —')
w_losses = widgets.Label(value='sync losses: —')
w_ab     = widgets.Label(value='ab_count: —')
w_gap    = widgets.Label(value='gap_per: —')
w_cam    = widgets.Label(value='cam_angle: —')

# Phase error plot
w_plot = widgets.Output()

err_history = []
running = False

def write_all(b):
    regs.write(0x08, w_kp.value)
    regs.write(0x0C, w_ki.value)
    regs.write(0x10, w_maxc.value)
    regs.write(0x14, w_phase.value)
    regs.write(0x18, w_tol.value)
    print(f'Written: KP={w_kp.value} KI={w_ki.value} MAXC={w_maxc.value} '
          f'PHASE={w_phase.value} TOL={w_tol.value}')

w_write.on_click(write_all)

def sign32(v):
    return v - 0x100000000 if v > 0x7FFFFFFF else v

def poll():
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    while running:
        try:
            status   = regs.read(0x24)
            sync_st  = status & 0x7
            tp       = regs.read(0x48)
            pe       = sign32(regs.read(0x54))
            co       = sign32(regs.read(0x58))
            sl       = regs.read(0x28)
            ab       = regs.read(0x44)
            gap      = regs.read(0x4C)
            cam      = regs.read(0x5C)

            rpm      = round(60_000_000 / (tp / 100) / 60) if tp > 0 else 0
            ph_deg   = pe / 4_294_967_296 * 360

            w_state.value  = f'sync: {STATES.get(sync_st, "?")}'  
            w_rpm.value    = f'RPM: {rpm}'
            w_ph_err.value = f'phase err: {ph_deg:.3f} deg'
            w_corr.value   = f'correction: {co}'
            w_losses.value = f'sync losses: {sl}'
            w_ab.value     = f'ab_count: {ab}'
            w_gap.value    = f'gap_per: {gap} ({gap/100000:.1f}ms)'
            w_cam.value    = f'cam_angle: {cam} ({cam/10:.1f} deg)'

            err_history.append(ph_deg)
            if len(err_history) > 60:
                err_history.pop(0)

            with w_plot:
                w_plot.clear_output(wait=True)
                fig, ax = plt.subplots(figsize=(8, 2))
                ax.plot(err_history, color='steelblue', linewidth=1.5)
                ax.axhline(0, color='gray', linewidth=0.5)
                ax.set_ylabel('deg')
                ax.set_title('Phase error history')
                ax.set_xlim(0, 60)
                plt.tight_layout()
                plt.show()
                plt.close()
        except Exception as e:
            print(f'Poll error: {e}')
        time.sleep(0.5)

# --- Layout ---
params_box  = widgets.VBox([w_kp, w_ki, w_maxc, w_phase, w_tol, w_write])
status_box  = widgets.VBox([w_state, w_rpm, w_ph_err, w_corr,
                             w_losses, w_ab, w_gap, w_cam])
top_row     = widgets.HBox([params_box, status_box])

display(top_row, w_plot)

# Start polling thread
running = True
t = threading.Thread(target=poll, daemon=True)
t.start()
print('Polling started. Run the stop cell to halt.')

Polling started. Run the stop cell to halt.


In [ ]:
# Run this cell to stop polling
running = False
print('Polling stopped')